In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.stats import ttest_rel, wilcoxon
from scipy.stats import f_oneway
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load Recommendations

In [ ]:
playlist_recommendation = pd.read_csv("dataset/recommendations_for_all_models_with_hybrid.csv")

In [ ]:
def convert_string_array_to_list(s):
    """Convert a string representation of an array into a list of integers."""
    if isinstance(s, str):  # Handle string case (where it's incorrectly stored)
        numbers = re.findall(r'\d+', s)  # Extract all numeric values
        return [int(x) for x in numbers]  # Convert to integers
    
    elif isinstance(s, np.ndarray):  # Handle NumPy array case
        return s.astype(int).tolist()
    
    elif isinstance(s, list):  # Handle lists with possible string numbers
        return [int(x) for x in s if str(x).isdigit()]
    
    return []  # Return empty list if the format is unexpected

# Apply function and debug output
playlist_recommendation['tracks_to_predict'] = playlist_recommendation['tracks_to_predict'].apply(lambda x: convert_string_array_to_list(x))
playlist_recommendation['random_recommendations'] = playlist_recommendation['random_recommendations'].apply(lambda x: convert_string_array_to_list(x))
playlist_recommendation['content_based_unweigted_recommendations'] = playlist_recommendation['content_based_unweigted_recommendations'].apply(lambda x: convert_string_array_to_list(x))
playlist_recommendation['content_based_weigted_recommendations'] = playlist_recommendation['content_based_weigted_recommendations'].apply(lambda x: convert_string_array_to_list(x))
playlist_recommendation['collaborative_filtering_recommendations'] = playlist_recommendation['collaborative_filtering_recommendations'].apply(lambda x: convert_string_array_to_list(x))
playlist_recommendation['hybrid_recommendations'] = playlist_recommendation['hybrid_recommendations'].apply(lambda x: convert_string_array_to_list(x))

playlist_recommendation.head()

# Code for Evaluation

In [ ]:
def compute_metrics_for_playlist(predicted_tracks, test_indices, k):
    top_k = predicted_tracks[:k]  # Consider only top K predictions

    # Hit@K
    hit = int(any(t in top_k for t in test_indices))

    # MRR and AP calculations
    precisions = []
    num_hits = 0
    mrr = 0.0

    for rank_idx, track_idx in enumerate(top_k):
        if track_idx in test_indices:
            num_hits += 1
            precision_at_k = num_hits / (rank_idx + 1)
            precisions.append(precision_at_k)
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)

    ap = np.mean(precisions) if precisions else 0.0

    return hit, mrr, ap

def evaluate_model(playlist, column_name, k):
    num_playlists = len(playlist)

    hit_total, mrr_total, ap_total = 0, 0, 0

    for idx, row in playlist.iterrows():
        test_indices = row['tracks_to_predict']
        predicted_tracks = row[column_name]

        hit, mrr, ap = compute_metrics_for_playlist(predicted_tracks, test_indices, k)
        
        hit_total += hit
        mrr_total += mrr
        ap_total += ap

    # Compute averages
    hit_ratio = hit_total / num_playlists if num_playlists > 0 else 0
    mrr_avg = mrr_total / num_playlists if num_playlists > 0 else 0
    map_avg = ap_total / num_playlists if num_playlists > 0 else 0

    return hit_ratio, mrr_avg, map_avg, column_name

# Level 4: Comparing Hybrid Model against Content Based Model & Collaborative Filtering

## Model Performance

Hybrid Model

In [ ]:
# Run evaluation
hit_ratio, mrr_avg, map_avg, column_name = evaluate_model(playlist_recommendation, 'hybrid_recommendations', k=50)

# Print results
print(f"Evaluation Metrics for {column_name}")
print(f"Hit@50: {hit_ratio:.4f}")
print(f"MRR: {mrr_avg:.4f}")
print(f"MAP@50: {map_avg:.4f}")

Content Based Model

In [ ]:
# Run evaluation
hit_ratio, mrr_avg, map_avg, column_name = evaluate_model(playlist_recommendation, 'content_based_unweigted_recommendations', k=50)

# Print results
print(f"Evaluation Metrics for {column_name}")
print(f"Hit@50: {hit_ratio:.4f}")
print(f"MRR: {mrr_avg:.4f}")
print(f"MAP@50: {map_avg:.4f}")

Collaborative Filtering

In [ ]:
# Run evaluation
hit_ratio, mrr_avg, map_avg, column_name = evaluate_model(playlist_recommendation, 'collaborative_filtering_recommendations', k=50)

# Print results
print(f"Evaluation Metrics for {column_name}")
print(f"Hit@50: {hit_ratio:.4f}")
print(f"MRR: {mrr_avg:.4f}")
print(f"MAP@50: {map_avg:.4f}")

## Hybrid Model vs Content Based

## Hybrid Model vs Collaborative Filtering

## Limitations with Accuracy Metrics

## New Behavioural Metrics

## Model Performance using Behavioural Metrics

### Load Datasets

In [ ]:
tracks = pd.read_csv("dataset/tracks_new.csv")
playlist_original = pd.read_csv("dataset/playlist_final_final.csv")

### Manipulation

In [ ]:
tracks['track_popularity'] = tracks['track_popularity'] / 100
tracks['track_idx'] = tracks['track_idx'].astype(int)
tracks = tracks.set_index('track_idx')
playlist_original['popularity_mean'] = playlist_original['popularity_mean'] / 100
playlist_original['playlist_idx'] = playlist_original['playlist_idx'].astype(int)
behavioural_scores = playlist_recommendation[['playlist_idx', 'cluster', 'tracks_to_predict']]

In [ ]:
tracks_relevant_columns = tracks[['track_popularity',
                                  'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                                  'Short', 'Medium', 'Long',
                                  "joy", "calm", "sadness", "fear", "energizing", "dreamy",
                                  "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack", "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)", "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"]]

In [ ]:
playlist_original['sentiment_centroid'] = playlist_original['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
playlist_original['genre_centroid'] = playlist_original['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Unpack the column into separate columns
playlist_original[['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']] = pd.DataFrame(playlist_original['genre_centroid'].tolist())

# Drop the original column if you no longer need it
playlist_original = playlist_original.drop(columns=['genre_centroid'])

# Unpack the column into separate columns
playlist_original[['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']] = pd.DataFrame(playlist_original['sentiment_centroid'].tolist())

# Drop the original column if you no longer need it
playlist_original = playlist_original.drop(columns=['sentiment_centroid'])

print(playlist_original)

playlist_relevant_columns = playlist_original[['popularity_mean', 
                                      'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion', 'era_2000s_proportion', 'era_modern_era_proportion',
                                      'length_short_proportion', 'length_medium_proportion', 'length_long_proportion',
                                      'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
                                      'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']]

playlist_original = playlist_original.set_index('playlist_idx')

### Code for Evaluation

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def avg_cosine_similarity(set_a, set_b, tracks_df, track_feat_columns):
    """
    Computes the average cosine similarity between all track pairs (a, b)
    where a ∈ set_a and b ∈ set_b, based on the provided track feature columns.
    """
    # Convert set_a and set_b to list if they are not already
    set_a = list(set_a)
    set_b = list(set_b)
    
    # Make sure that the track indices are valid
    a_vectors = tracks_df.loc[set_a, track_feat_columns].values
    b_vectors = tracks_df.loc[set_b, track_feat_columns].values
    
    if a_vectors.shape[0] == 0 or b_vectors.shape[0] == 0:
        return 0.0  # Avoid similarity calculation if vectors are empty

    # Compute the cosine similarity between all pairs of tracks in a_vectors and b_vectors
    sim_matrix = cosine_similarity(a_vectors, b_vectors)
    return sim_matrix.mean()

# Example: To apply this on each playlist
def compute_relevance_for_playlist(row, tracks_df, track_feat_columns):
    true_tracks = row['tracks_to_predict']
    recommended_tracks = row['hybrid_recommendations']
    
    # Calculate cosine similarity for the top k recommended tracks and true tracks
    sim_score = avg_cosine_similarity(recommended_tracks, true_tracks, tracks_df, track_feat_columns)
    return sim_score

def calculate_diversity_score(recommended_tracks, tracks_df, track_feat_columns):
    """
    Calculate the diversity score for a given set of recommended tracks.
    The diversity score is based on the average pairwise cosine similarity between recommended tracks.
    A lower cosine similarity indicates higher diversity.
    """
    # Extract the feature vectors of the recommended tracks
    recommended_vectors = tracks_df.loc[recommended_tracks, track_feat_columns].values
    
    if recommended_vectors.shape[0] == 0:
        return 0.0  # If there are no recommended tracks, return a default score
    
    # Compute the pairwise cosine similarity between all recommended tracks
    sim_matrix = cosine_similarity(recommended_vectors)
    
    # We remove the diagonal of the similarity matrix, as it represents self-similarity
    np.fill_diagonal(sim_matrix, 0)  # Set the diagonal to 0 to ignore self-similarity
    
    # Calculate the average of the off-diagonal similarities
    avg_similarity = sim_matrix.sum() / (sim_matrix.shape[0] * (sim_matrix.shape[0] - 1))
    
    # Diversity score: lower cosine similarity means higher diversity
    diversity_score = 1 - avg_similarity  # You can also directly return avg_similarity if you prefer
    
    return diversity_score

# Example: Apply this to each playlist
def compute_diversity_for_playlist(row, tracks_df, track_feat_columns):
    recommended_tracks = row['hybrid_recommendations']
    
    # Calculate the diversity score for the recommended tracks
    diversity_score = calculate_diversity_score(recommended_tracks, tracks_df, track_feat_columns)
    return diversity_score

def compute_novelty(row, playlist_relevant_df, tracks_df, playlist_feat_columns, track_feat_columns):
    # Get playlist feature vector (assume index of row matches that in playlist_relevant_df)
    playlist_vector = playlist_relevant_df.loc[row.name, playlist_feat_columns].values.reshape(1, -1)

    # Get track IDs for recommended tracks
    recommended_ids = row['hybrid_recommendations']
    
    # Make sure it's a list of valid indices
    recommended_ids = list(recommended_ids)
    
    # Get feature vectors for recommended tracks
    track_vectors = tracks_df.loc[recommended_ids, track_feat_columns].values
    
    if track_vectors.shape[0] == 0:
        return 0.0  # Avoid errors on empty recommended lists
    
    # Compute cosine similarity between playlist and each recommended track
    sim_scores = cosine_similarity(playlist_vector, track_vectors)[0]  # Shape (n_tracks,)
    
    # Novelty = 1 - average similarity
    novelty_score = 1 - np.mean(sim_scores)
    return novelty_score

### Model Performance

In [ ]:
# Apply this to the dataset and create a new column in the `playlist_recommendation` DataFrame
behavioural_scores['relevance'] = playlist_recommendation.apply(compute_relevance_for_playlist, axis=1, 
                                               tracks_df=tracks, track_feat_columns=tracks_relevant_columns.columns)

print("Behaviorial Metrics of Hybrid Model:\n")

# Print the updated DataFrame
print("Mean relevance:", behavioural_scores['relevance'].mean())

# Apply this function to the playlist_recommendation DataFrame to get the diversity score for each playlist
behavioural_scores['diversity'] = playlist_recommendation.apply(compute_diversity_for_playlist, axis=1, 
                                          tracks_df=tracks, track_feat_columns=tracks_relevant_columns.columns)

# Print the dataframe with the diversity scores
print("Mean diversity:", behavioural_scores['diversity'].mean())

# Align indices
playlist_recommendation = playlist_recommendation.reset_index(drop=True)
playlist_original = playlist_original.reset_index(drop=True)

behavioural_scores['novelty'] = playlist_recommendation.apply(
    compute_novelty,
    axis=1,
    playlist_relevant_df=playlist_original,
    tracks_df=tracks,
    playlist_feat_columns=playlist_relevant_columns.columns,
    track_feat_columns=tracks_relevant_columns.columns
)

behavioural_scores['serendipity'] = behavioural_scores['relevance'] * behavioural_scores['novelty']

print("Mean serendipity:", behavioural_scores['serendipity'].mean())

# Level 5: Comparing Hybrid Model for Different Clusters

We have shown in Level 3 that our segmentation is effective. 

We will continue to explore if there are any significant differences using the behavioural metrics too, and the systematic reasons behind it.

In [ ]:
metrics_df = pd.DataFrame({
    'Avg Relevance': behavioural_scores.groupby('cluster')['relevance'].mean(),
    'Avg Diversity': behavioural_scores.groupby('cluster')['diversity'].mean(),
    'Avg Serendipity': behavioural_scores.groupby('cluster')['serendipity'].mean()
})

print("Evaluation Metrics for Hybrid Model\n")

# Assuming metrics_df is already defined as before
for cluster in metrics_df.index:
    rel = metrics_df.loc[cluster, "Avg Relevance"]
    div = metrics_df.loc[cluster, "Avg Diversity"]
    ser = metrics_df.loc[cluster, "Avg Serendipity"]

    print(f"Metrics for Cluster {cluster}:")
    print(f"Avg Relevance: {rel:.4f}")
    print(f"Avg Diversity: {div:.4f}")
    print(f"Avg Serendipity: {ser:.4f}\n")